# Eliminar outliers (sale y rent)

Reglas aplicadas sobre copias:
1. Regla fija por umbral (la que pediste).
2. Regla adicional de 3 desviaciones estandar (3 sigma) en `precio` y `superficie_construida_m2`, iterativa hasta que no queden casos > 3 sigma.

In [1]:
from pathlib import Path

import pandas as pd

def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "processed" / "idealistaAPI").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto con data/processed/idealistaAPI")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "idealistaAPI"

SALE_PATH = DATA_DIR / "sale_homes_clean.csv"
RENT_PATH = DATA_DIR / "rent_homes_clean.csv"

SALE_OUT_PATH = DATA_DIR / "sale_homes_clean_outliers.csv"
RENT_OUT_PATH = DATA_DIR / "rent_homes_clean_outliers.csv"

sale_df = pd.read_csv(SALE_PATH)
rent_df = pd.read_csv(RENT_PATH)

print(f"Sale original: {sale_df.shape}")
print(f"Rent original: {rent_df.shape}")

Sale original: (604, 20)
Rent original: (493, 20)


In [2]:
def remove_sigma_iterative(df: pd.DataFrame, price_col: str, surface_col: str, sigma: float = 3.0):
    out = df.copy()
    removed_total = 0

    while True:
        mu_p = out[price_col].mean()
        sd_p = out[price_col].std(ddof=0)
        mu_s = out[surface_col].mean()
        sd_s = out[surface_col].std(ddof=0)

        if (sd_p == 0) or pd.isna(sd_p):
            mask_price = pd.Series(False, index=out.index)
        else:
            mask_price = ((out[price_col] - mu_p).abs() / sd_p) > sigma

        if (sd_s == 0) or pd.isna(sd_s):
            mask_surface = pd.Series(False, index=out.index)
        else:
            mask_surface = ((out[surface_col] - mu_s).abs() / sd_s) > sigma

        mask_sigma = mask_price | mask_surface
        n = int(mask_sigma.sum())

        if n == 0:
            break

        removed_total += n
        out = out.loc[~mask_sigma].copy()

    return out, removed_total


def clean_dataset(df: pd.DataFrame, price_limit: float, surface_limit: float, label: str):
    work = df.copy()

    # 1) Regla fija solicitada
    fixed_mask = (work["superficie_construida_m2"] > surface_limit) & (work["precio"] > price_limit)
    fixed_removed = int(fixed_mask.sum())
    work = work.loc[~fixed_mask].copy()

    # 2) Regla adicional de 3 sigma (precio y superficie)
    work, sigma_removed = remove_sigma_iterative(
        work,
        price_col="precio",
        surface_col="superficie_construida_m2",
        sigma=3.0,
    )

    # Verificaciones
    fixed_left = int(((work["superficie_construida_m2"] > surface_limit) & (work["precio"] > price_limit)).sum())

    mu_p = work["precio"].mean()
    sd_p = work["precio"].std(ddof=0)
    mu_s = work["superficie_construida_m2"].mean()
    sd_s = work["superficie_construida_m2"].std(ddof=0)

    sigma_left_price = 0 if (sd_p == 0 or pd.isna(sd_p)) else int((((work["precio"] - mu_p).abs() / sd_p) > 3).sum())
    sigma_left_surface = 0 if (sd_s == 0 or pd.isna(sd_s)) else int((((work["superficie_construida_m2"] - mu_s).abs() / sd_s) > 3).sum())

    print(f"\n===== {label} =====")
    print(f"Regla fija eliminados: {fixed_removed}")
    print(f"Regla 3 sigma eliminados: {sigma_removed}")
    print(f"Total final filas: {work.shape[0]}")
    print(f"Verificacion regla fija restantes: {fixed_left}")
    print(f"Verificacion >3 sigma precio restantes: {sigma_left_price}")
    print(f"Verificacion >3 sigma superficie restantes: {sigma_left_surface}")
    print(f"Max precio final: {work['precio'].max()}")
    print(f"Max superficie final: {work['superficie_construida_m2'].max()}")

    return work


sale_clean = clean_dataset(sale_df, price_limit=1_400_000, surface_limit=750, label="SALE")
rent_clean = clean_dataset(rent_df, price_limit=3_900, surface_limit=300, label="RENT")


===== SALE =====
Regla fija eliminados: 2
Regla 3 sigma eliminados: 203
Total final filas: 399
Verificacion regla fija restantes: 0
Verificacion >3 sigma precio restantes: 0
Verificacion >3 sigma superficie restantes: 0
Max precio final: 425000.0
Max superficie final: 143.0

===== RENT =====
Regla fija eliminados: 3
Regla 3 sigma eliminados: 78
Total final filas: 412
Verificacion regla fija restantes: 0
Verificacion >3 sigma precio restantes: 0
Verificacion >3 sigma superficie restantes: 0
Max precio final: 1800.0
Max superficie final: 161.0


In [3]:
sale_clean.to_csv(SALE_OUT_PATH, index=False)
rent_clean.to_csv(RENT_OUT_PATH, index=False)

# Comprobacion final leyendo los CSV exportados
sale_check = pd.read_csv(SALE_OUT_PATH)
rent_check = pd.read_csv(RENT_OUT_PATH)

sale_fixed_left = int(((sale_check["superficie_construida_m2"] > 750) & (sale_check["precio"] > 1_400_000)).sum())
rent_fixed_left = int(((rent_check["superficie_construida_m2"] > 300) & (rent_check["precio"] > 3_900)).sum())

print(f"\nCSV exportado: {SALE_OUT_PATH}")
print(f"CSV exportado: {RENT_OUT_PATH}")
print(f"Filas SALE exportado: {sale_check.shape[0]}")
print(f"Filas RENT exportado: {rent_check.shape[0]}")
print(f"Verificacion final regla fija SALE restantes: {sale_fixed_left}")
print(f"Verificacion final regla fija RENT restantes: {rent_fixed_left}")

assert sale_fixed_left == 0, "Quedan outliers de regla fija en SALE"
assert rent_fixed_left == 0, "Quedan outliers de regla fija en RENT"
print("Verificacion OK: exportacion correcta y sin outliers de regla fija.")


CSV exportado: /Users/sitomachucas/Documents/BezanillaSL/data/processed/idealistaAPI/sale_homes_clean_outliers.csv
CSV exportado: /Users/sitomachucas/Documents/BezanillaSL/data/processed/idealistaAPI/rent_homes_clean_outliers.csv
Filas SALE exportado: 399
Filas RENT exportado: 412
Verificacion final regla fija SALE restantes: 0
Verificacion final regla fija RENT restantes: 0
Verificacion OK: exportacion correcta y sin outliers de regla fija.
